# NB05: Curated & Specialized Evidence (Tiers 3–4)

**Purpose**: Extract evidence from curated and specialized sources:
1. **PaperBLAST curatedgene** (255K rows) — EC numbers from `desc` field
2. **FitnessBrowser seedclass** (61.9K rows) — EC numbers from `num` where `type=1`
3. **FitnessBrowser besthitmetacyc** (59.7K rows) — MetaCyc rxnId + ecnum
4. **InterPro→GO→EC** (1.18B protein2ipr → 30K go_mapping) — indirect domain chain

RAST (Channel 8) was already processed in NB02 as `rast_protein_ec.parquet`.

**Output**: `curated_evidence_ec.parquet` (combined protein/locus-EC pairs from Tiers 3-4)

**Requires**: BERDL JupyterHub (Spark session), NB02 bridge tables

In [1]:
import os, re
import pandas as pd

try:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
except ImportError:
    from get_spark_session import get_spark_session

spark = get_spark_session()
spark.sql("SET spark.sql.autoBroadcastJoinThreshold = -1")

DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
metacyc_bridge = pd.read_parquet(f'{DATA_DIR}/metacyc_to_reaction.parquet')
bridge_ecs = set(ec_bridge['ec'])
bridge_metacyc = set(metacyc_bridge['metacyc_reaction'])

print(f'EC bridge:      {len(ec_bridge):,} mappings, {len(bridge_ecs):,} unique ECs')
print(f'MetaCyc bridge: {len(metacyc_bridge):,} mappings, {len(bridge_metacyc):,} unique MetaCyc rxns')
print(f'Spark session ready.')

EC bridge:      22,823 mappings, 6,100 unique ECs
MetaCyc bridge: 5,039 mappings, 4,480 unique MetaCyc rxns
Spark session ready.


## 1. PaperBLAST curatedgene

The `desc` field contains function descriptions with embedded EC numbers.
Extract EC from text using regex. The `db` column indicates curation source
(SwissProt, BRENDA, metacyc, CharProtDB, etc.).

In [2]:
paperblast_df = spark.sql("""
    SELECT db, protId, id2, name, desc
    FROM kescience_paperblast.curatedgene
""").toPandas()

print(f'PaperBLAST curatedgene: {len(paperblast_df):,} rows')
print(f'\nSource databases:')
print(paperblast_df['db'].value_counts().to_string())
print(f'\nSample with desc:')
for _, row in paperblast_df[paperblast_df['desc'].str.contains(r'\d+\.\d+\.\d+', na=False)].head(5).iterrows():
    print(f'  {row["db"]:12s} {row["protId"]:20s} {row["desc"][:80]}')

PaperBLAST curatedgene: 255,096 rows

Source databases:
db
SwissProt     110171
biolip         42571
BRENDA         33012
metacyc        12700
REBASE         12388
ENA             9251
CAZy            8878
TCDB            8509
CharProtDB      8021
ecocyc          4198
regprecise      3159
reanno          1885
prodoric         353

Sample with desc:


  SwissProt    Q6V4H0               8-hydroxygeraniol dehydrogenase; Cr10HGO; EC 1.1.1.324
  SwissProt    P0DO85               Strychnine-10-hydroxylase; Snv10H; CYP450 monooxygenase 10H; Cytochrome P450 10H
  SwissProt    P0DO87               Strychnine-11-hydroxylase; Snv11H; CYP450 monooxygenase 11H; Cytochrome P450 11H
  SwissProt    P21215               12alpha-hydroxysteroid dehydrogenase; HSDH; EC 1.1.1.176
  SwissProt    Q92AT0               1,2-beta-oligoglucan phosphorylase; EC 2.4.1.333


In [3]:
ec_pattern = re.compile(r'(\d+\.\d+\.\d+\.\d+|\d+\.\d+\.\d+\.-|\d+\.\d+\.-\.-|\d+\.-\.-\.-)')

rows = []
for _, r in paperblast_df.iterrows():
    desc = r['desc'] if pd.notna(r['desc']) else ''
    ecs = ec_pattern.findall(desc)
    for ec in ecs:
        rows.append({'protein': r['protId'], 'ec': ec, 'source_db': r['db']})

paperblast_ec = pd.DataFrame(rows).drop_duplicates(subset=['protein', 'ec'])

print(f'PaperBLAST EC extractions: {len(paperblast_ec):,} protein-EC pairs')
print(f'  Unique proteins: {paperblast_ec.protein.nunique():,}')
print(f'  Unique ECs:      {paperblast_ec.ec.nunique():,}')

pb_ecs = set(paperblast_ec['ec'])
print(f'\n  Matching bridge: {len(pb_ecs & bridge_ecs):,}')
print(f'  Not in bridge:   {len(pb_ecs - bridge_ecs):,}')

pb_rxns = set(ec_bridge[ec_bridge['ec'].isin(pb_ecs)]['rxn_bare'])
print(f'  Balanced reactions reachable: {len(pb_rxns):,}')

PaperBLAST EC extractions: 96,586 protein-EC pairs
  Unique proteins: 84,248
  Unique ECs:      6,447

  Matching bridge: 4,700
  Not in bridge:   1,747
  Balanced reactions reachable: 16,138


## 2. FitnessBrowser seedclass

The `type` column = 1 for EC numbers, stored in `num` column.
The `orgId` + `locusId` identifies the gene.

In [4]:
seedclass_df = spark.sql("""
    SELECT orgId, locusId, type, num
    FROM kescience_fitnessbrowser.seedclass
""").toPandas()

print(f'seedclass total rows: {len(seedclass_df):,}')
print(f'\nType distribution:')
print(seedclass_df['type'].value_counts().to_string())

seedclass_ec = seedclass_df[seedclass_df['type'] == '1'].copy()
seedclass_ec = seedclass_ec.rename(columns={'num': 'ec'})
seedclass_ec['locus'] = seedclass_ec['orgId'] + ':' + seedclass_ec['locusId']
seedclass_ec = seedclass_ec[['locus', 'ec']].drop_duplicates()

print(f'\nseedclass EC rows (type=1): {len(seedclass_ec):,}')
print(f'  Unique loci: {seedclass_ec.locus.nunique():,}')
print(f'  Unique ECs:  {seedclass_ec.ec.nunique():,}')

sc_ecs = set(seedclass_ec['ec'])
print(f'\n  Matching bridge: {len(sc_ecs & bridge_ecs):,}')
print(f'  Not in bridge:   {len(sc_ecs - bridge_ecs):,}')

sc_rxns = set(ec_bridge[ec_bridge['ec'].isin(sc_ecs)]['rxn_bare'])
print(f'  Balanced reactions reachable: {len(sc_rxns):,}')

seedclass total rows: 61,874

Type distribution:
type
1    56820
2     5054

seedclass EC rows (type=1): 56,820
  Unique loci: 53,431
  Unique ECs:  1,446

  Matching bridge: 1,260
  Not in bridge:   186
  Balanced reactions reachable: 8,828


## 3. FitnessBrowser besthitmetacyc

Two evidence paths:
- `rxnId`: direct MetaCyc reaction IDs → bridge via `metacyc_to_reaction.parquet`
- `ecnum`: EC numbers → bridge via `ec_to_reaction.parquet`

The `rxnId` can contain multiple reactions per row (same locusId appears in multiple rows).

In [5]:
bhmc_df = spark.sql("""
    SELECT orgId, locusId, protId, identity, rxnId, ecnum
    FROM kescience_fitnessbrowser.besthitmetacyc
""").toPandas()

print(f'besthitmetacyc total rows: {len(bhmc_df):,}')
print(f'  Unique loci:     {bhmc_df.apply(lambda r: f"{r.orgId}:{r.locusId}", axis=1).nunique():,}')
print(f'  Unique MetaCyc protIds: {bhmc_df.protId.nunique():,}')
print(f'  Rows with rxnId: {bhmc_df.rxnId.notna().sum():,}')
print(f'  Rows with ecnum: {bhmc_df.ecnum.notna().sum():,}')

print(f'\nSample rows:')
for _, row in bhmc_df.head(5).iterrows():
    print(f'  {row["orgId"]}:{row["locusId"]} rxnId={row["rxnId"]} ecnum={row["ecnum"]}')

besthitmetacyc total rows: 59,723


  Unique loci:     39,097
  Unique MetaCyc protIds: 4,724
  Rows with rxnId: 54,949
  Rows with ecnum: 46,766

Sample rows:
  acidovorax_3H11:Ac3H11_10 rxnId=RXN-11834 ecnum=5.4.99.20
  acidovorax_3H11:Ac3H11_13 rxnId=PREPHENATEDEHYDRAT-RXN ecnum=4.2.1.51
  acidovorax_3H11:Ac3H11_13 rxnId=CARBOXYCYCLOHEXADIENYL-DEHYDRATASE-RXN ecnum=4.2.1.91
  acidovorax_3H11:Ac3H11_29 rxnId=3-DEHYDROQUINATE-DEHYDRATASE-RXN ecnum=4.2.1.10
  acidovorax_3H11:Ac3H11_30 rxnId=AMINOCYL-TRNA-HYDROLASE-RXN ecnum=3.1.1.29


In [6]:
bhmc_rxnid = bhmc_df[bhmc_df['rxnId'].notna() & (bhmc_df['rxnId'] != '')].copy()
bhmc_rxnid['locus'] = bhmc_rxnid['orgId'] + ':' + bhmc_rxnid['locusId']
bhmc_rxnid_pairs = bhmc_rxnid[['locus', 'rxnId']].drop_duplicates()

bhmc_metacyc_ids = set(bhmc_rxnid_pairs['rxnId'])
print(f'besthitmetacyc unique MetaCyc rxnIds: {len(bhmc_metacyc_ids):,}')
print(f'  Matching MetaCyc bridge: {len(bhmc_metacyc_ids & bridge_metacyc):,}')
print(f'  Not in bridge:           {len(bhmc_metacyc_ids - bridge_metacyc):,}')

bhmc_via_metacyc = set(metacyc_bridge[metacyc_bridge['metacyc_reaction'].isin(bhmc_metacyc_ids)]['rxn_bare'])
print(f'  Balanced reactions reachable via rxnId: {len(bhmc_via_metacyc):,}')

besthitmetacyc unique MetaCyc rxnIds: 4,054
  Matching MetaCyc bridge: 597
  Not in bridge:           3,457
  Balanced reactions reachable via rxnId: 950


In [7]:
bhmc_ec = bhmc_df[bhmc_df['ecnum'].notna() & (bhmc_df['ecnum'] != '')].copy()
bhmc_ec['locus'] = bhmc_ec['orgId'] + ':' + bhmc_ec['locusId']
bhmc_ec_pairs = bhmc_ec[['locus', 'ecnum']].rename(columns={'ecnum': 'ec'}).drop_duplicates()

bhmc_ec_set = set(bhmc_ec_pairs['ec'])
print(f'besthitmetacyc unique ECs: {len(bhmc_ec_set):,}')
print(f'  Matching EC bridge: {len(bhmc_ec_set & bridge_ecs):,}')
print(f'  Not in bridge:      {len(bhmc_ec_set - bridge_ecs):,}')

bhmc_via_ec = set(ec_bridge[ec_bridge['ec'].isin(bhmc_ec_set)]['rxn_bare'])
print(f'  Balanced reactions reachable via ecnum: {len(bhmc_via_ec):,}')

bhmc_combined = bhmc_via_metacyc | bhmc_via_ec
print(f'\nbesthitmetacyc combined reactions: {len(bhmc_combined):,}')
print(f'  rxnId-only:  {len(bhmc_via_metacyc - bhmc_via_ec):,}')
print(f'  ecnum-only:  {len(bhmc_via_ec - bhmc_via_metacyc):,}')
print(f'  Both:        {len(bhmc_via_metacyc & bhmc_via_ec):,}')

besthitmetacyc unique ECs: 2,228
  Matching EC bridge: 1,677
  Not in bridge:      551
  Balanced reactions reachable via ecnum: 7,112

besthitmetacyc combined reactions: 7,210
  rxnId-only:  98
  ecnum-only:  6,260
  Both:        852


## 4. InterPro → GO → EC (Tier 4)

Chain: `protein2ipr` (1.18B) → `go_mapping` (30K ipr→GO) → GO→EC.

The GO→EC mapping is not directly in BERDL. We use `interproscan_go` from the pangenome
to get gene_cluster→GO, then check if UniProt `identifier` GO xrefs overlap with known
EC-bearing GO terms. Alternatively, we can extract EC-associated GO terms from the GO
consortium's ec2go mapping.

Since this is Tier 4 (indirect, low confidence), we focus on quantifying coverage
rather than building a complete mapping.

In [8]:
go_mapping = spark.sql("""
    SELECT ipr_id, go_id, go_name
    FROM kescience_interpro.go_mapping
""").toPandas()

print(f'InterPro go_mapping: {len(go_mapping):,} ipr→GO mappings')
print(f'  Unique IPR IDs: {go_mapping.ipr_id.nunique():,}')
print(f'  Unique GO IDs:  {go_mapping.go_id.nunique():,}')

mf_terms = go_mapping[go_mapping['go_name'].str.contains('activity|ase|transferase|synthase|kinase|lyase|isomerase|ligase', case=False, na=False)]
print(f'\n  GO terms with enzymatic activity keywords: {len(mf_terms):,}')
print(f'  Sample enzymatic GO terms:')
for _, row in mf_terms.head(5).iterrows():
    print(f'    {row["go_id"]} {row["go_name"]}')

InterPro go_mapping: 30,200 ipr→GO mappings
  Unique IPR IDs: 14,799
  Unique GO IDs:  5,633

  GO terms with enzymatic activity keywords: 9,640
  Sample enzymatic GO terms:
    GO:0003707 nuclear steroid receptor activity
    GO:0019888 protein phosphatase regulator activity
    GO:0000159 protein phosphatase type 2A complex
    GO:0004869 cysteine-type endopeptidase inhibitor activity
    GO:0008641 ubiquitin-like modifier activating enzyme activity


In [9]:
ipr_go_ec = spark.sql("""
    SELECT DISTINCT g.ipr_id, g.go_id, u.xref AS ec
    FROM kescience_interpro.go_mapping g
    JOIN (
        SELECT DISTINCT
            REPLACE(entity_id, 'uniprot:', '') AS protein,
            xref
        FROM refdata_uniprot.identifier
        WHERE db = 'EC'
    ) u ON 1=1
    WHERE false
""").toPandas()

print('InterPro→GO→EC chain requires an external GO→EC mapping (ec2go from GO consortium).')
print('This mapping is not available in BERDL.')
print()
print('Alternative: use interproscan_go from the pangenome, which already links')
print('gene_clusters to GO terms. But still need GO→EC bridge.')
print()

interproscan_go_count = spark.sql("""
    SELECT COUNT(*) as n FROM kbase_ke_pangenome.interproscan_go
""").collect()[0]['n']

interproscan_go_unique_go = spark.sql("""
    SELECT COUNT(DISTINCT go_id) as n FROM kbase_ke_pangenome.interproscan_go
""").collect()[0]['n']

print(f'interproscan_go: {interproscan_go_count:,} gene_cluster-GO pairs')
print(f'  Unique GO terms: {interproscan_go_unique_go:,}')
print()
print('DECISION: Defer InterPro→GO→EC chain to future work.')
print('Reason: Requires external ec2go mapping not in BERDL, and this is the')
print('lowest-confidence evidence tier. Coverage ceiling estimate only.')

InterPro→GO→EC chain requires an external GO→EC mapping (ec2go from GO consortium).
This mapping is not available in BERDL.

Alternative: use interproscan_go from the pangenome, which already links
gene_clusters to GO terms. But still need GO→EC bridge.



interproscan_go: 266,317,724 gene_cluster-GO pairs
  Unique GO terms: 7,641

DECISION: Defer InterPro→GO→EC chain to future work.
Reason: Requires external ec2go mapping not in BERDL, and this is the
lowest-confidence evidence tier. Coverage ceiling estimate only.


## 5. Combine Tier 3 Evidence & Coverage Summary

Note: Tier 3 sources use different ID spaces (PaperBLAST protIds, FitnessBrowser loci).
We track EC sets for reaction coverage, not cross-source protein merges.

In [10]:
balanced_ids = set(
    pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t', usecols=['id', 'status'])
    .query("status == 'OK'")['id']
    .str.replace('seed.reaction:', '', regex=False)
)

print(f'Balanced reactions: {len(balanced_ids):,}')
print()

print('Tier 3 Reaction Coverage:')
print(f'  PaperBLAST EC:           {len(pb_rxns):,} / {len(balanced_ids):,} ({100*len(pb_rxns)/len(balanced_ids):.1f}%)')
print(f'  seedclass EC:            {len(sc_rxns):,} / {len(balanced_ids):,} ({100*len(sc_rxns)/len(balanced_ids):.1f}%)')
print(f'  besthitmetacyc rxnId:    {len(bhmc_via_metacyc):,} / {len(balanced_ids):,} ({100*len(bhmc_via_metacyc)/len(balanced_ids):.1f}%)')
print(f'  besthitmetacyc ecnum:    {len(bhmc_via_ec):,} / {len(balanced_ids):,} ({100*len(bhmc_via_ec)/len(balanced_ids):.1f}%)')
print(f'  besthitmetacyc combined: {len(bhmc_combined):,} / {len(balanced_ids):,} ({100*len(bhmc_combined)/len(balanced_ids):.1f}%)')

tier3_rxns = pb_rxns | sc_rxns | bhmc_combined
print(f'\n  Tier 3 combined:         {len(tier3_rxns):,} / {len(balanced_ids):,} ({100*len(tier3_rxns)/len(balanced_ids):.1f}%)')

print(f'\nAdditive value:')
print(f'  seedclass beyond PaperBLAST:   {len(sc_rxns - pb_rxns):,}')
print(f'  besthitmetacyc beyond both:    {len(bhmc_combined - pb_rxns - sc_rxns):,}')

Balanced reactions: 34,343

Tier 3 Reaction Coverage:
  PaperBLAST EC:           16,138 / 34,343 (47.0%)
  seedclass EC:            8,828 / 34,343 (25.7%)
  besthitmetacyc rxnId:    950 / 34,343 (2.8%)
  besthitmetacyc ecnum:    7,112 / 34,343 (20.7%)
  besthitmetacyc combined: 7,210 / 34,343 (21.0%)

  Tier 3 combined:         16,290 / 34,343 (47.4%)

Additive value:
  seedclass beyond PaperBLAST:   87
  besthitmetacyc beyond both:    65


In [11]:
pb_filtered = paperblast_ec[paperblast_ec['ec'].isin(bridge_ecs)].copy()
pb_filtered['channel'] = 'paperblast'
pb_filtered = pb_filtered[['protein', 'ec', 'channel']].drop_duplicates()

print(f'PaperBLAST bridge-matched: {len(pb_filtered):,} protein-EC pairs')
print(f'  Unique proteins: {pb_filtered.protein.nunique():,}')
print(f'  Unique ECs:      {pb_filtered.ec.nunique():,}')

PaperBLAST bridge-matched: 74,911 protein-EC pairs
  Unique proteins: 66,338
  Unique ECs:      4,700


In [12]:
sc_filtered = seedclass_ec[seedclass_ec['ec'].isin(bridge_ecs)].copy()
sc_filtered = sc_filtered.rename(columns={'locus': 'protein'})
sc_filtered['channel'] = 'seedclass'
sc_filtered = sc_filtered[['protein', 'ec', 'channel']].drop_duplicates()

print(f'seedclass bridge-matched: {len(sc_filtered):,} locus-EC pairs')
print(f'  Unique loci: {sc_filtered.protein.nunique():,}')
print(f'  Unique ECs:  {sc_filtered.ec.nunique():,}')

seedclass bridge-matched: 49,930 locus-EC pairs
  Unique loci: 46,881
  Unique ECs:  1,260


In [13]:
bhmc_ec_filtered = bhmc_ec_pairs[bhmc_ec_pairs['ec'].isin(bridge_ecs)].copy()
bhmc_ec_filtered = bhmc_ec_filtered.rename(columns={'locus': 'protein'})
bhmc_ec_filtered['channel'] = 'besthitmetacyc_ec'
bhmc_ec_filtered = bhmc_ec_filtered[['protein', 'ec', 'channel']].drop_duplicates()

print(f'besthitmetacyc EC bridge-matched: {len(bhmc_ec_filtered):,} locus-EC pairs')

bhmc_rxnid_matched = bhmc_rxnid_pairs.merge(
    metacyc_bridge[['metacyc_reaction', 'rxn_bare']],
    left_on='rxnId', right_on='metacyc_reaction', how='inner'
)
print(f'besthitmetacyc rxnId bridge-matched: {len(bhmc_rxnid_matched):,} locus-reaction pairs')
print(f'  Unique loci:      {bhmc_rxnid_matched.locus.nunique():,}')
print(f'  Unique reactions: {bhmc_rxnid_matched.rxn_bare.nunique():,}')

besthitmetacyc EC bridge-matched: 25,817 locus-EC pairs
besthitmetacyc rxnId bridge-matched: 22,310 locus-reaction pairs
  Unique loci:      10,077
  Unique reactions: 950


In [14]:
curated_ec = pd.concat([
    pb_filtered[['protein', 'ec', 'channel']],
    sc_filtered[['protein', 'ec', 'channel']],
    bhmc_ec_filtered[['protein', 'ec', 'channel']]
], ignore_index=True).drop_duplicates()

curated_ec.to_parquet(f'{DATA_DIR}/curated_evidence_ec.parquet', index=False)
print(f'Saved curated_evidence_ec.parquet: {len(curated_ec):,} protein/locus-EC pairs')
print(f'  Channels: {curated_ec.channel.value_counts().to_dict()}')

bhmc_rxnid_matched[['locus', 'rxnId', 'rxn_bare']].to_parquet(
    f'{DATA_DIR}/besthitmetacyc_rxnid.parquet', index=False
)
print(f'Saved besthitmetacyc_rxnid.parquet: {len(bhmc_rxnid_matched):,} locus-reaction pairs')

Saved curated_evidence_ec.parquet: 150,658 protein/locus-EC pairs
  Channels: {'paperblast': 74911, 'seedclass': 49930, 'besthitmetacyc_ec': 25817}
Saved besthitmetacyc_rxnid.parquet: 22,310 locus-reaction pairs


## 6. Summary

In [15]:
print('=' * 60)
print('NB05 TIER 3-4 EVIDENCE SUMMARY')
print('=' * 60)
print(f'\nTier 3 Channels:')
print(f'  1. PaperBLAST:     {pb_filtered.protein.nunique():,} proteins, {pb_filtered.ec.nunique():,} ECs -> {len(pb_rxns):,} reactions')
print(f'  2. seedclass:      {sc_filtered.protein.nunique():,} loci, {sc_filtered.ec.nunique():,} ECs -> {len(sc_rxns):,} reactions')
print(f'  3. besthitmetacyc: {bhmc_ec_filtered.protein.nunique():,} loci (EC) + {bhmc_rxnid_matched.locus.nunique():,} loci (rxnId) -> {len(bhmc_combined):,} reactions')
print(f'\nTier 4:')
print(f'  InterPro->GO->EC: DEFERRED (no GO->EC bridge in BERDL)')
print(f'\nCombined Tier 3: {len(tier3_rxns):,} / {len(balanced_ids):,} balanced reactions ({100*len(tier3_rxns)/len(balanced_ids):.1f}%)')
print(f'\nSaved:')
print(f'  curated_evidence_ec.parquet ({len(curated_ec):,} pairs)')
print(f'  besthitmetacyc_rxnid.parquet ({len(bhmc_rxnid_matched):,} pairs)')
print(f'\nNext: NB06 -- evidence integration & scoring')

NB05 TIER 3-4 EVIDENCE SUMMARY

Tier 3 Channels:
  1. PaperBLAST:     66,338 proteins, 4,700 ECs -> 16,138 reactions


  2. seedclass:      46,881 loci, 1,260 ECs -> 8,828 reactions
  3. besthitmetacyc: 22,640 loci (EC) + 10,077 loci (rxnId) -> 7,210 reactions

Tier 4:
  InterPro->GO->EC: DEFERRED (no GO->EC bridge in BERDL)

Combined Tier 3: 16,290 / 34,343 balanced reactions (47.4%)

Saved:
  curated_evidence_ec.parquet (150,658 pairs)
  besthitmetacyc_rxnid.parquet (22,310 pairs)

Next: NB06 -- evidence integration & scoring
